# Audit graph representatives with pynauty

Install `hecke-mass[graphs]` in this kernel's environment. From a checkout, use `python -m pip install './python[graphs]'`.

There are 64 labeled simple graphs on four vertices. Use pynauty's canonical certificate to choose one representative of each isomorphism class.

In [ ]:
from itertools import combinations

import pynauty
from hecke_mass.adapters import audit_pynauty_graphs

n = 4
edges = list(combinations(range(n), 2))
representatives = {}
for mask in range(1 << len(edges)):
    graph = pynauty.Graph(n)
    for index, (u, v) in enumerate(edges):
        if mask & (1 << index):
            graph.connect_vertex(u, [v])
    representatives.setdefault(pynauty.certificate(graph), graph)

assert len(representatives) == 11
report = audit_pynauty_graphs(iter(representatives.values()), n)
assert (report.status, report.covered, report.deficit) == ("complete", 64, 0)
report

## Drop the empty graph

The empty graph has a stabilizer of order $4!$. Removing its orbit drops exactly one labeled graph.

In [ ]:
empty_certificate = pynauty.certificate(pynauty.Graph(n))
incomplete = (
    graph for key, graph in representatives.items() if key != empty_certificate
)
report = audit_pynauty_graphs(incomplete, n)
assert (report.status, report.covered, report.deficit) == ("missing", 63, 1)
report

The count uses the automorphism orders of the supplied representatives. Coverage certifies a complete list when the representatives are distinct isomorphism classes and all lie in the counted family. Here the certificates establish the distinctness of the rows.